# Microsoft Fabric — Pipeline Validator v3.0
# Lakehouse → Warehouse Reconciliation (Corrected)

> **Author:** Myla Ram Reddy  
> **Platform:** Microsoft Fabric (Runtime 1.3 — Spark 3.5, Delta 3.2)

---

## Errors Fixed in This Version

| Error | Root Cause | Fix Applied |
|-------|-----------|-------------|
| `UnsupportedOperationException: No default context found` | No Lakehouse attached to Notebook | Attach Lakehouse via left panel before running |
| `FabricSparkTDSReadError: Either source is invalid` | `synapsesql` used to READ Lakehouse — hits SQL endpoint which has sync lag | Use `spark.sql()` or Delta read for Lakehouse |
| `TypeError: __init__() got unexpected keyword argument 'lakehouse_name'` | `PipelineValidator` was built around `synapsesql` — wrong design | Redesigned to accept DataFrames directly |

---

## Correct Connector Usage Rules

```
LAKEHOUSE READ  →  spark.sql()  or  spark.read.format("delta")     ✅
                   NOT synapsesql  (SQL endpoint has sync lag)      ❌

LAKEHOUSE WRITE →  df.write.format("delta").saveAsTable()           ✅
                   NOT synapsesql  (write-to-warehouse only)        ❌

WAREHOUSE WRITE →  df.write.mode("overwrite").synapsesql("wh.dbo.t") ✅

WAREHOUSE READ  →  spark.read.synapsesql("wh.dbo.table")           ✅
```

---

## PRE-REQUISITE — Attach Lakehouse to This Notebook

Before running any cell:

```
Left panel of Notebook
  └── Lakehouses
       └── + Add lakehouse
            └── Existing lakehouse
                 └── select "sales_lakehouse"  → Add
```

You should see `sales_lakehouse` appear in the left panel with a pin icon (default).  
Then **stop any running session** before proceeding.

---

In [ ]:
# Cell 1 — Configuration
# Update these to match your Fabric workspace items

LAKEHOUSE_NAME   = "sales_lakehouse"   # Lakehouse item name
WAREHOUSE_NAME   = "sales_warehouse"   # Warehouse item name
TABLE_NAME       = "sales_data"        # Table name (no schema prefix)
WAREHOUSE_SCHEMA = "dbo"               # Warehouse schema
KEY_COLS         = ["order_id"]        # Columns for duplicate check
NUMERIC_COLS     = ["amount",          # Columns for numeric reconciliation
                    "quantity"]

# Full warehouse reference used by synapsesql
WAREHOUSE_TABLE  = f"{WAREHOUSE_NAME}.{WAREHOUSE_SCHEMA}.{TABLE_NAME}"

print(f"Lakehouse : {LAKEHOUSE_NAME}")
print(f"Warehouse : {WAREHOUSE_NAME}")
print(f"Table     : {TABLE_NAME}")
print(f"WH ref    : {WAREHOUSE_TABLE}")

In [ ]:
# Cell 2 — Imports
# synapsesql connector is pre-installed in Fabric Runtime 1.3

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

from pipeline_validator import (
    PipelineValidator,
    row_count_check,
    column_count_check,
    schema_match_check,
    null_check,
    duplicate_check,
    numeric_reconciliation,
    column_value_distribution,
)

print("All imports successful")

---

## STEP 1 — Create Sample Data and Save to Lakehouse

We use `saveAsTable()` to write Delta files to the Lakehouse.  
No schema prefix — the default attached Lakehouse is used automatically.

In [ ]:
# Cell 3 — Create sample sales data with intentional DQ issues

from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, DateType
)
from datetime import date

sample_data = [
    (1001, "Alice",   "Electronics", 1500.00, 3, date(2024, 1,  5)),
    (1002, "Bob",     "Clothing",     250.50, 2, date(2024, 1,  6)),
    (1003, "Charlie", "Electronics", 3200.00, 1, date(2024, 1,  7)),
    (1004, "Diana",   "Furniture",   8500.00, 1, date(2024, 1,  8)),
    (1005, "Alice",   "Electronics", 1500.00, 3, date(2024, 1,  5)),  # duplicate
    (1006, "Eve",     "Clothing",     None,   1, date(2024, 1,  9)),  # null amount
    (1007, "Frank",   "Furniture",   4200.00, 2, date(2024, 1, 10)),
    (1008, "Grace",   "Electronics",  980.00, 4, date(2024, 1, 11)),
    (1009, "Hank",    "Clothing",     310.00, 3, date(2024, 1, 12)),
    (1010, "Iris",    "Furniture",   6100.00, 1, date(2024, 1, 13)),
]

schema = StructType([
    StructField("order_id",   IntegerType(), False),
    StructField("customer",   StringType(),  True),
    StructField("category",   StringType(),  True),
    StructField("amount",     DoubleType(),  True),
    StructField("quantity",   IntegerType(), True),
    StructField("order_date", DateType(),    True),
])

df = spark.createDataFrame(sample_data, schema)

# Write to Lakehouse as Delta table
# TABLE_NAME only — no lakehouse prefix, no schema prefix
# The default attached Lakehouse resolves the context
df.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

print(f"Saved {df.count()} rows to Lakehouse → {TABLE_NAME}")
df.show(truncate=False)

---

## STEP 2 — Read from Lakehouse

Use `spark.sql()` — reads Delta files **directly**.  
No SQL endpoint, no sync lag, no `synapsesql` needed here.

In [ ]:
# Cell 4 — Read from Lakehouse using spark.sql()
# This reads the Delta files directly — fast, reliable, no sync lag

lakehouse_df = spark.sql(f"SELECT * FROM {TABLE_NAME}")

# Alternative — direct Delta path (equally correct)
# lakehouse_df = spark.read.format("delta").load(f"Tables/{TABLE_NAME}")

print(f"Lakehouse row count : {lakehouse_df.count()}")
print(f"Lakehouse columns   : {lakehouse_df.columns}")
lakehouse_df.printSchema()

---

## STEP 3 — Write to Warehouse using `synapsesql`

`synapsesql` is correct here — writing to a Warehouse.  
Internally uses `COPY INTO` — staged, scalable, production-grade.

In [ ]:
# Cell 5 — Write Lakehouse DataFrame → Warehouse using synapsesql
# synapsesql internally: DataFrame → staging storage → COPY INTO → Warehouse

print(f"Writing to Warehouse : {WAREHOUSE_TABLE}")
print(f"Save mode            : overwrite")
print(f"Mechanism            : COPY INTO (staged)")

lakehouse_df.write.mode("overwrite").synapsesql(WAREHOUSE_TABLE)

print(f"\nWrite complete → {WAREHOUSE_TABLE}")

---

## STEP 4 — Read Back from Warehouse using `synapsesql`

`synapsesql` is correct here — reading from a Warehouse SQL engine.

In [ ]:
# Cell 6 — Read from Warehouse using synapsesql
# This is the TARGET DataFrame for reconciliation

print(f"Reading from Warehouse : {WAREHOUSE_TABLE}")

warehouse_df = spark.read.synapsesql(WAREHOUSE_TABLE)

print(f"Warehouse row count : {warehouse_df.count()}")
print(f"Warehouse columns   : {warehouse_df.columns}")
warehouse_df.printSchema()

---

## STEP 5 — Run Pipeline Validation

`PipelineValidator` now takes **DataFrames directly** — no `lakehouse_name`,  
no `warehouse_name`, no `table_name` arguments.  
You load the DataFrames yourself, then pass them in.

In [ ]:
# Cell 7 — Create PipelineValidator and run all 6 checks
#
# NEW signature (v3.0):
#   PipelineValidator(
#       source_df    = <lakehouse DataFrame>,
#       target_df    = <warehouse DataFrame>,
#       source_label = "display name for source",
#       target_label = "display name for target",
#       key_cols     = ["col_used_for_duplicate_check"],
#       numeric_cols = ["col1", "col2"],
#   )
#
# No spark, no lakehouse_name, no warehouse_name — just the DataFrames

pv = PipelineValidator(
    source_df    = lakehouse_df,
    target_df    = warehouse_df,
    source_label = "Sales Lakehouse",
    target_label = "Sales Warehouse",
    key_cols     = KEY_COLS,
    numeric_cols = NUMERIC_COLS,
)

pv.run_all_checks()

---

## STEP 6 — Run Individual Checks (Optional)

You can also call each standalone function directly.

In [ ]:
# Cell 8 — Individual standalone function calls

print("--- Row count ---")
print(row_count_check(lakehouse_df, warehouse_df))

print("\n--- Column count ---")
print(column_count_check(lakehouse_df, warehouse_df))

print("\n--- Schema match ---")
print(schema_match_check(lakehouse_df, warehouse_df))

print("\n--- Nulls (Lakehouse) ---")
print(null_check(lakehouse_df, "Lakehouse"))

print("\n--- Duplicates (Lakehouse) ---")
print(duplicate_check(lakehouse_df, key_cols=["order_id"], label="Lakehouse"))

print("\n--- Numeric reconciliation ---")
nr = numeric_reconciliation(lakehouse_df, warehouse_df, ["amount", "quantity"])
print("Passed:", nr["passed"])

In [ ]:
# Cell 9 — Column value distribution (category QA check)

for label, df in [("Lakehouse", lakehouse_df), ("Warehouse", warehouse_df)]:
    dist = column_value_distribution(df, "category", label, top_n=5)
    print(f"\nTop categories — {label}:")
    for val, count in dist["top_values"]:
        print(f"   {val:<20} : {count} rows")

---

## STEP 7 — Write Audit Results to Lakehouse

In [ ]:
# Cell 10 — Write validation results to Lakehouse audit log table

from pyspark.sql.types import StructType, StructField, StringType, BooleanType

summary = pv.get_summary()

check_map = {
    "row_count"             : summary["results"].get("row_count",   {}).get("passed"),
    "column_count"          : summary["results"].get("column_count",{}).get("passed"),
    "schema_match"          : summary["results"].get("schema",      {}).get("passed"),
    "nulls_source"          : summary["results"].get("nulls",       {}).get("source",{}).get("passed"),
    "nulls_target"          : summary["results"].get("nulls",       {}).get("target",{}).get("passed"),
    "duplicates_source"     : summary["results"].get("duplicates",  {}).get("source",{}).get("passed"),
    "duplicates_target"     : summary["results"].get("duplicates",  {}).get("target",{}).get("passed"),
    "numeric_reconciliation": summary["results"].get("numeric",     {}).get("passed"),
}

audit_rows = [
    (
        summary["run_time"],
        summary["source_label"],
        summary["target_label"],
        check_name,
        bool(passed) if passed is not None else False
    )
    for check_name, passed in check_map.items()
]

audit_schema = StructType([
    StructField("run_time",   StringType(),  True),
    StructField("source",     StringType(),  True),
    StructField("target",     StringType(),  True),
    StructField("check_name", StringType(),  True),
    StructField("passed",     BooleanType(), True),
])

audit_df = spark.createDataFrame(audit_rows, audit_schema)

# Append to audit log table in Lakehouse
audit_df.write.format("delta").mode("append").saveAsTable("pipeline_audit_log")

print("Audit log written to Lakehouse → pipeline_audit_log")
audit_df.show(truncate=False)

---

## Error Reference — All Three Errors and Their Fixes

### Error 1
```
UnsupportedOperationException:
No default context found, please attach a lakehouse before running
spark sql queries with partial namespaces
```
**Fix:** Attach Lakehouse via left panel → `+ Add lakehouse` → pin as default → stop session → re-run.

---

### Error 2
```
Py4JJavaError:
FabricSparkTDSReadError: Either source is invalid or user doesn't have
read access. Reference - b2026_lakehouse.dbo.sales_data
```
**Root cause:** `synapsesql` was used to read from the Lakehouse.  
It routes through the SQL Analytics Endpoint which has sync lag — the table
exists in Delta but the SQL endpoint hasn't seen it yet.

**Fix:**
```python
# WRONG
df = spark.read.synapsesql("sales_lakehouse.dbo.sales_data")

# CORRECT
df = spark.sql("SELECT * FROM sales_data")
# or
df = spark.read.format("delta").load("Tables/sales_data")
```

---

### Error 3
```
TypeError:
PipelineValidator.__init__() got an unexpected keyword argument 'lakehouse_name'
```
**Root cause:** Old `PipelineValidator` was designed around `synapsesql` reads from Lakehouse.
Since that approach is wrong, the whole constructor was redesigned.

**Fix — new constructor signature:**
```python
# WRONG (v1 and v2)
pv = PipelineValidator(
    spark          = spark,
    lakehouse_name = "sales_lakehouse",   # ← caused TypeError
    warehouse_name = "sales_warehouse",
    table_name     = "sales_data",
)

# CORRECT (v3)
pv = PipelineValidator(
    source_df    = lakehouse_df,          # DataFrame you loaded yourself
    target_df    = warehouse_df,          # DataFrame you loaded yourself
    source_label = "Sales Lakehouse",
    target_label = "Sales Warehouse",
    key_cols     = ["order_id"],
    numeric_cols = ["amount", "quantity"],
)
```

---
*Notebook v3.0 — Myla Ram Reddy | All three real-world Fabric errors resolved*